In [ ]:
#import libraries
import numpy as np
import pandas as pd

%matplotlib inline
import matplotlib.pyplot as plt

## Reading the data and adding the column headers

In [ ]:
# declare file path and read data
path = '../Data/imports-85.data'
df = pd.read_csv(path, header=None)

In [ ]:
# display first 10 lines
df.head(10)

In [ ]:
# create headers list (Refer to file: ../Data/imports-85.names)
headers = ["symboling","normalized-losses","make","fuel-type","aspiration", "num-of-doors","body-style",
         "drive-wheels","engine-location","wheel-base", "length","width","height","curb-weight","engine-type",
         "num-of-cylinders", "engine-size","fuel-system","bore","stroke","compression-ratio","horsepower",
         "peak-rpm","city-mpg","highway-mpg","price"]

In [ ]:
# insert column headers
df.columns=headers
df.head(10)

## Identifying and handling missing data values

In [ ]:
# Replace the "?" with NaN 
df.replace("?", np.nan, inplace=True)
df.head(10)

In [ ]:
#Check for missing data
missing_data = df.isnull()

In [ ]:
# Count missing values in each column
for column in missing_data.columns.values.tolist():
    print(column)
    print(missing_data[column].value_counts())
    print("")

### Dealing with missing values

In [ ]:
# Calculate the mean value for the "normalised-losses" column.
avg_norm_loss = df["normalized-losses"].astype("float").mean(axis=0)
print("Mean of normalized losses:", avg_norm_loss)

In [ ]:
# Replace "nan" with mean value in "normalised_losses" column
df["normalized-losses"].replace(np.nan,avg_norm_loss, inplace=True)

In [ ]:
# Calculate the mean value for the "bore" column and replace the "nan" values
avg_bore = df["bore"].astype("float").mean(axis=0)
print("Mean of bore:", avg_bore)
df["bore"].replace(np.nan,avg_bore, inplace=True)

In [ ]:
# Calculate the mean value in "stroke" and replace the "nan" values
avg_stroke = df["stroke"].astype("float").mean(axis=0)
print("Mean of stroke:", avg_stroke)
df["stroke"].replace(np.nan, avg_stroke, inplace=True)

In [ ]:
# Calculate the mean value in "horsepower" and replace the "nan" values
avg_hp = df["horsepower"].astype("float").mean(axis=0)
print("Mean of horsepower:", avg_hp)
df["horsepower"].replace(np.nan, avg_hp, inplace=True)

In [ ]:
# Calculate the mean value of "peak-rpm" and replace the "nan"values
avg_peak = df["peak-rpm"].astype("float").mean(axis=0)
print("Mean peak-rpm:", avg_peak)
df["peak-rpm"].replace(np.nan, avg_peak, inplace=True)

In [ ]:
# Check the values in the "num-of-doors" column
df["num-of-doors"].value_counts()

In [ ]:
# Alternatively check which is the most frequent value in "num-of-doors" column
df["num-of-doors"].value_counts().idxmax()

In [ ]:
# Replace the "nan" values in the "num-of-doors" column with the most frequent
df["num-of-doors"].replace(np.nan, "four", inplace=True)

In [ ]:
# Drop all rows that do not have price data
df.dropna(subset=["price"], axis=0, inplace=True)

In [ ]:
df.head(10)

## Correcting data format

In [ ]:
# Check data types for each column
df.dtypes

In [ ]:
# Convert data types to proper format
df[["normalized-losses"]] = df[["normalized-losses"]].astype("int")
df[["bore", "stroke", "peak-rpm", "price"]] = df[["bore","stroke","peak-rpm","price"]].astype("float")

In [ ]:
df.dtypes

## Data standardization
Standardization is the process of transforming data into a common format, allowing the researcher to make meaningful comparisons.

In [ ]:
# Convert mpg ot L/100km (=> 235/mpg)
df[["city-mpg", "highway-mpg"]] = 235/df[["city-mpg","highway-mpg"]]

# Rename the transformed columns
df.rename(columns={"city-mpg":"city-L/100km","highway-mpg":"highway-L/100km"}, inplace=True)

In [ ]:
# Check your transformed data
df.head()

## Data Normalization
Normalization is the process of transforming values of several variables into a similar range.

In [ ]:
# Scale columns "length", "width", "height" so that they range from 0 to 1
# (replace original value by original value/maximum value)
df["width"]  = df["width"]/df["width"].max()
df["length"] = df["length"]/df["length"].max()
df["height"] = df["height"]/df["height"].max()

# Check the scaled data
df[["length", "width", "height"]].head()

## Binning
It is the process of tranforming continous numerical variables into discrete categorical 'bins' for grouped analysis.

In [ ]:
# We'll use horsepower as an example here
# Convert to correct format
df["horsepower"] = df["horsepower"].astype(int, copy=True)

# plot the histogram of horsepower
plt.hist(df["horsepower"])
plt.xlabel("horsepower")
plt.ylabel("count")
plt.title("Horsepower bins")
plt.show()

In [ ]:
# Set the bins to 3
bins = np.linspace(min(df["horsepower"]), max(df["horsepower"]), 4)
group_names = ["Low", "Medium", "High"]

# Use the "cut" function to determine what each value of "horsepower" belongs to
df["horsepower-binned"] = pd.cut(df["horsepower"], bins, labels=group_names, include_lowest=True)
df[["horsepower", "horsepower-binned"]].head(20)

In [ ]:
# Let's see the number of vehicles in each bin
df["horsepower-binned"].value_counts()

In [ ]:
# Plot the distributon of each bin
plt.bar(group_names, df["horsepower-binned"].value_counts())
plt.xlabel("horsepower")
plt.ylabel("count")
plt.title("horsepower bins")
plt.show()

In [ ]:
# Plot a histogram to visualize the bins
plt.hist(df["horsepower"], bins=3)
plt.xlabel("horsepower")
plt.ylabel("count")
plt.title("horsepower bins")
plt.show()

## Indicator (Dummy) variables
An indicator variable is a numerical variable used to label categories. They are called dummies because the themselves numbers don't have inherent meanings.
Indicator variables are used so that we can perform regression analysis on categorical data.

In [ ]:
df.columns

In [ ]:
# Get the dummy variables and assign them to dataa frame "dummy_variable_1".
dummy_variable_1 = pd.get_dummies(df["fuel-type"])
dummy_variable_1.head()

In [ ]:
# Change the column names for clarity
dummy_variable_1.rename(columns={"gas":"fuel-type-gas", "diesel":"fuel-type-diesel"}, inplace=True)
dummy_variable_1.head()

In [ ]:
# Merge dataframe "df" and "dummy-variable_1"
df = pd.concat([df, dummy_variable_1], axis=1)

# Drop original column "fuel-type" from 'df'
df.drop("fuel-type", axis=1, inplace=True)
df.head()

In [ ]:
# Implement indicator variable for the column aspiration
dummy_variable_2 = pd.get_dummies(df["aspiration"])
dummy_variable_2.rename(columns={"std":"aspiration-std", "turbo":"aspiration-turbo"}, inplace=True)

# Merge dataframe with the dummies and drop the aspiration column
df = pd.concat([df, dummy_variable_2], axis=1)
df.drop("aspiration", axis=1, inplace=True)
df.head()

In [ ]:
# Save the clean data to a new csv-type file
df.to_csv("../Data/clean_df.csv")